# MediCheck-GPT: run the full flow on Colab
Runtime → Change runtime type → **T4 GPU**. Run top to bottom. Every number in the README comes from this notebook's output.

## 0. Setup: GPU, code, data

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv   # confirm the T4 is attached

In [ ]:
from google.colab import drive
drive.mount('/content/drive')                     # the raw CMS file lives in my Drive
ROOT = '/content/drive/MyDrive/build-transformer'   # same folder as 01_data.ipynb

In [ ]:
!git clone https://github.com/cece15/medicheck-gpt.git /content/medicheck-gpt   # private repo: Colab will ask for a GitHub token
%cd /content/medicheck-gpt
!pip install -q -r requirements.txt

## 1. Data: CSV → one sentence per row → split by hospital

In [ ]:
!python gpt.py --input "$ROOT/data/raw/MUP_INP_RY26_P03_V10_DY24_PrvSvc.CSV" --prepare data/processed
!head -c 600 data/processed/train.txt

## 2. Baseline: bigram (one character of memory)

In [ ]:
!python gpt.py --input data/processed --train checkpoints/bigram.pth --model bigram --context_size 8 --lr 1e-2 --steps 3000

## 3. Transformer: 4 layers, 4 heads, 128-dim, context 271 (one full record)
Cosine LR with warmup, weight decay, early stopping. First run (3 Nov) overfit after ~1,500 steps, so these guards matter.

In [ ]:
!python gpt.py --input data/processed --train checkpoints/gpt.pth --steps 10000 --eval_every 500 --patience 4

In [ ]:
# Loss curve for the slides
import json, matplotlib.pyplot as plt
h = json.load(open('checkpoints/gpt_history.json'))
plt.plot([r['step'] for r in h], [r['train'] for r in h], label='train')
plt.plot([r['step'] for r in h], [r['val'] for r in h], label='validation (unseen hospitals)')
plt.xlabel('step'); plt.ylabel('cross-entropy loss'); plt.legend(); plt.title('MediCheck-GPT training')
plt.savefig('checkpoints/loss_curve.png', dpi=200, bbox_inches='tight'); plt.show()

## 3a. Hyperparameter comparison (Week 7: fine-tuning and hyperparameter tuning)
Same data, same seed, same early stopping. Only the model size and dropout change. About 15 minutes.

In [ ]:
# smaller and larger than the base model (base = 4 layers, 4 heads, 128-dim, already trained above)
!python gpt.py --input data/processed --train checkpoints/gpt_small.pth --n_layer 2 --n_head 2 --n_embd 64 --steps 6000
!python gpt.py --input data/processed --train checkpoints/gpt_large.pth --n_layer 6 --n_head 6 --n_embd 192 --dropout 0.2 --steps 6000

In [ ]:
# one table for the slides: which size generalizes best to unseen hospitals?
import math, torch, pandas as pd
from src.model import build, count_params
rows = []
for name in ['gpt_small', 'gpt', 'gpt_large']:
    ck = torch.load(f'checkpoints/{name}.pth', map_location='cpu')
    best = min(ck['history'], key=lambda r: r['val'])            # the checkpoint holds the best evaluation
    rows.append({'model': name, **{k: ck['config'][k] for k in ['n_layer', 'n_head', 'n_embd', 'dropout']},
                 'parameters': count_params(build(ck['kind'], **ck['config'])),
                 'best_step': best['step'], 'best_val_loss': round(best['val'], 4),
                 'val_perplexity': round(math.exp(best['val']), 3), 'seconds': best['seconds']})
pd.DataFrame(rows)

## 3b. Kacper's recommendation: an LLM teacher writes plain-English descriptions
Python computes every number; the LLM only phrases. Any sentence with a number not in the record is rejected. About 20–30 minutes on a T4.

In [ ]:
!python gpt.py --input data/processed --teacher data/described --limit 3000
!head -c 900 data/described/train.txt

## 3c. Student: train a small GPT to explain records the way the teacher does
Metrics (perplexity, ROUGE-L on validation) print at every evaluation, from inside the training loop.

In [ ]:
!python gpt.py --input data/described --train checkpoints/explain.pth --steps 10000 --eval_every 500 --patience 4

In [ ]:
!python gpt.py --eval checkpoints/explain.pth --prompt "$(sed -n 2p data/processed/test.txt)"

## 4. Generate

In [ ]:
!python gpt.py --eval checkpoints/gpt.pth --n 3
!python gpt.py --eval checkpoints/gpt.pth --n 2 --prompt "Trinity Health Livonia Hospital in Livonia, MI billed DRG "

## 5. Evaluate: perplexity, token accuracy, ROUGE-L (+ BLEU, BERTScore), record validity

In [ ]:
!python gpt.py --input data/processed --evaluate checkpoints/bigram.pth
!python gpt.py --input data/processed --evaluate checkpoints/gpt.pth --hf_metrics
!python gpt.py --input data/described --evaluate checkpoints/explain.pth --hf_metrics

## 6. Flag silent errors: plant four error types in unseen hospitals' records, measure ROC AUC

In [ ]:
!python gpt.py --input data/processed --flag checkpoints/bigram.pth
!python gpt.py --input data/processed --flag checkpoints/gpt.pth

## 7. Save results to Drive (checkpoints are too big for GitHub)

In [ ]:
!mkdir -p "$ROOT/results" && cp data/described/teacher_report.json checkpoints/*.json checkpoints/*.png checkpoints/gpt.pth checkpoints/explain.pth "$ROOT/results/" && ls "$ROOT/results"

## Live demo (Tuesday)
`!python gpt.py --eval checkpoints/explain.pth --interactive`  (paste a record, get the plain-English sentence)

`!python gpt.py --flag checkpoints/gpt.pth --flag_file suspicious.txt`  (rank records, most surprising first)